In [2]:
# =========================
# Colab: Limpieza + Normalización de documentos legales (solo DOCX)
# -> Salida: JSON limpio (.json) 1:1 por archivo de entrada + sidecar baseline .txt + diffs opcionales
# Salidas en /content/output/clean_json, /content/output/raw_baseline y /content/output/debug_diffs
# v1.29-legal-peru-rag-json (parches añadidos sobre v1.28)
# Cambios clave vs v1.28:
#  - [v1.29 ADD] Normalización OCR de montos: Z→2, I→1 en contextos monetarios + tracking de montos antes/después (money_raw_set / money_norm_set)
#  - [v1.29 ADD] Compactación robusta de dígitos “tipo cheque” (secuencias largas con espacios/saltos), sin tocar líneas sensibles
#  - [v1.29 ADD] Diccionario OCR en tablas ampliado + fuzzy (>90) para nombres/cargos en celdas tabulares
#  - [v1.29 ADD] Conservar tablas 2×2 si contienen claves de actas (ACTA, D.N.I., Notificador, FIRMA, NEGATIVA, etc.) con meta kept_small_acta=True
#  - [v1.29 ADD] PII en tablas: activado en columnas etiquetadas y con RUC enmascarado (TABLE_PII_ONLY=True, MASK_RUC=True por defecto v1.29)
#  - [v1.29 ADD] Golden tests (opcionales) para tokens conocidos; sólo registran pass/fail en transform_log
# =========================

!pip -q install python-docx regex dateparser rapidfuzz unidecode ftfy

import os, io, re, json, uuid, unicodedata, hashlib, difflib
from typing import List, Dict, Tuple, Optional

import dateparser
from unidecode import unidecode
from rapidfuzz import fuzz, process as fuzzprocess  # [v1.29 ADD]
from ftfy import fix_text

# ---------- Config ----------
PIPELINE_VERSION = "v1.29-legal-peru-rag-json"   # (antes: v1.28-legal-peru-rag-json)
OUTPUT_DIR = "/content/output"
JSON_DIR = os.path.join(OUTPUT_DIR, "clean_json")
RAW_BASELINE_DIR = os.path.join(OUTPUT_DIR, "raw_baseline")
DIFF_DIR = os.path.join(OUTPUT_DIR, "debug_diffs")
os.makedirs(JSON_DIR, exist_ok=True)
os.makedirs(RAW_BASELINE_DIR, exist_ok=True)
os.makedirs(DIFF_DIR, exist_ok=True)

ACCEPTED_EXTS = {".docx"}

# Toggles (conservadores por defecto)
ENABLE_PII_MASKING = False
MASK_RUC = True                 # [v1.29 ADD] activar enmascarado de RUC en tablas
FORMAT_CURRENCY_AMOUNTS = True
MERGE_SPLIT_WATERMARKS = True
DEDUP_INSTITUTIONAL_HEADERS = True
STRICT_NO_LOSS_RATIO = 0.985
ABORT_ON_LOSS = False
INCLUDE_BASELINE_RAW = True
REMOVE_LINE_NUMBERS = True

# PII sólo en tablas etiquetadas (DNI/RUC) sin tocar texto corrido
TABLE_PII_ONLY = True           # [v1.29 ADD] activar enmascarado SOLO en tablas etiquetadas

# Heurística de logs
VERBOSE = True
EMOJI_OK = "✅"
EMOJI_DO = "🧹"
EMOJI_WARN = "⚠️"
EMOJI_INFO = "ℹ️"

SECTION_HEADERS = [
    "ENCABEZADO",
    "VISTO", "Visto",
    "CONSIDERANDO", "Considerando",
    "BASE LEGAL", "Base Legal",
    "ANTECEDENTES", "Antecedentes",
    "FUNDAMENTOS", "Fundamentos", "FUNDAMENTO", "Fundamento",
    "ANÁLISIS", "ANALISIS", "Análisis", "Analisis",
    "PARTE EXPOSITIVA", "Parte Expositiva",
    "SE RESUELVE", "RESUELVE", "Se Resuelve"
]
SECTION_HEADERS_UP = [h.upper() for h in SECTION_HEADERS]

DEFAULT_ENTITY_CANDIDATES = [
    "SATH", "SAT HUANCAYO", "SERVICIO DE ADMINISTRACIÓN TRIBUTARIA DE HUANCAYO",
    "MUNICIPALIDAD PROVINCIAL DE TACNA", "MUNICIPALIDAD PROVINCIAL DE HUANCAYO",
    "MUNICIPALIDAD", "GERENCIA",
    "SUNAT", "GOBIERNO REGIONAL", "MTC", "POLICÍA NACIONAL", "PNP",
    "MINISTERIO PÚBLICO", "PODER JUDICIAL", "RENIEC", "ESSALUD", "SBS", "SUCAMEC"
]

# Correcciones comunes (OCR/typos)
COMMON_FIXES = {
    r"\bBolognesl\b": "Bolognesi",
    r"\bTr[áa]nsito\b": "Tránsito",
    r"\bAnalisis\b": "Análisis",
    r"\bArticulo\b": "Artículo",
    r"\bResolucion\b": "Resolución",
    r"\bGerenc[aá]a\b": "Gerencia",
    r"\bS A T H\b": "SATH",
    r"\bLey\s+N\s*[Oº]\b": "Ley N°",
    r"(\d)\s+(\d)": r"\1\2",

    # Fixes OCR en mayúsculas
    r"\bCOMPETENFC\b": "COMPETENTE",
    r"\bSANCICNADOR\b": "SANCIONADOR",
    r"\bDISPUES?TLO\b": "DISPUESTO",
    r"\bDISPUESLO\b": "DISPUESTO",
    r"\bDEI\b": "DEL",
    r"\bSON\s+PLACA\b": "SIN PLACA",

    r"\(\s*sin\s+placa\s*\)": "(SIN PLACA)",

    r"\b(\d{2})-(\d{3})(\d{6,})\b": r"\1-\2-\3",

    r"(?i)\bN\s*O\s*(?=\d)": "N° ",
    r"(?i)\bNro\.?\s*(?=\d)": "N° ",
    r"(?i)\bNo\.?\s*(?=\d)": "N° ",
    r"(?i)\bNº\s*(?=\d)": "N° ",

    r"(?i)\bArt\.?\s*(?=\d)": "Artículo ",
}

# Aliases de normas -> canon
NORM_ALIASES = [
    (r"\bD\W*S\W*\.?\b", "DS"),
    (r"\bD\W*U\W*\.?\b", "DU"),
    (r"\bD\W*L\W*\.?\b", "DL"),
    (r"\bDECRETO\s+SUPREMO\b", "DS"),
    (r"\bDECRETO\s+DE\s+URGENCIA\b", "DU"),
    (r"\bDECRETO\s+LEGISLATIVO\b", "DL"),
    (r"\bLEY\s+N[°º]\b", "LEY N°"),
]

ACCENT_FIXES = {
    "transito":"tránsito","resolucion":"resolución","analisis":"análisis","articulo":"artículo",
    "administracion":"administración","direccion":"dirección","mision":"misión","vision":"visión",
    "gerencia":"gerencia","subgerencia":"subgerencia","publico":"público","publica":"pública",
    "regimen":"régimen","peru":"perú","codigo":"código","numero":"número",
    "competencia":"competencia","procedimiento":"procedimiento","sancion":"sanción"
}

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

from docx import Document
from docx.table import Table
from docx.text.paragraph import Paragraph
from docx.oxml.table import CT_Tbl
from docx.oxml.text.paragraph import CT_P

def log(msg: str, emoji: str = EMOJI_INFO):
    if VERBOSE:
        print(f"{emoji} {msg}")

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def slugify(text: str) -> str:
    t = unicodedata.normalize("NFKD", text)
    t = re.sub(r"[^\w\s-]", "", t, flags=re.UNICODE)
    t = re.sub(r"\s+", "-", t).strip("-").lower()
    return t[:80] if t else f"doc-{uuid.uuid4().hex[:8]}"

def nfkc(text: str) -> str:
    return unicodedata.normalize("NFKC", text)

def _apply_and_count(text: str, pattern: str, repl: str, flags=re.I) -> Tuple[str, int]:
    new_text, n = re.subn(pattern, repl, text, flags=flags)
    return new_text, n

# ------- Sensibilidad de líneas / contexto -------
SENSITIVE_RX = re.compile(r"\b(OFICIO|RESOLUCI[ÓO]N|EXPEDIENTE|CHEQUE|PAPELETA|BOLETA|FACTURA)\b", re.I)
def is_sensitive_line(s: str) -> bool:
    return bool(SENSITIVE_RX.search(s or ""))

# ------- Watermarks -------
WM_TERMS = [
    r"cam\s*scann?er\w*",
    r"camscanner",
    r"scanned\s*with\s*cam\s*scann?er",
    r"adobe\s*scan\w*",
    r"genius\s*scan\w*",
    r"(?:microsoft|office)\s*lens\w*",
    r"tiny\s*scanner\w*",
    r"scan\s*bot\w*",
    r"text\s*fairy\w*",
    r"scanner\s*pro\w*",
    r"prizmo\w*",
    r"notebloc\w*",
    r"docu\s*scan\w*",
    r"fast\s*scanner\w*",
    r"pdf\s*scanner\w*",
    r"clear\s*scan\w*",
]
WM_LINE = re.compile(r"(?i)^\s*(?:scann?\s*ed\s*with|escanead[oa]\s+con)?\s*(?:" + "|".join(WM_TERMS) + r")\b.*$")
WM_INLINE = re.compile(r"(?i)(?:scann?\s*ed\s*with|escanead[oa]\s+con)?\s*(?:" + "|".join(WM_TERMS) + r")\b")
def is_watermark_line(s: str) -> bool:
    return bool(WM_LINE.search(s or ""))
def remove_watermark_substrings(s: str) -> Tuple[str, int]:
    count = 0
    def _rm(m):
        nonlocal count
        count += 1
        return ""
    cleaned = WM_INLINE.sub(_rm, s or "")
    cleaned = re.sub(r"\s{2,}", " ", cleaned).strip()
    return cleaned, count
def merge_split_watermark_lines(text: str) -> Tuple[str, int]:
    lines = text.splitlines()
    out = []
    i = 0
    merged = 0
    join_pat = re.compile(r"(?i)\b(cam\s*scann?er|adobe\s*scan|microsoft\s*lens|office\s*lens|genius\s*scan|tiny\s*scanner|scan\s*bot|pdf\s*scanner|clear\s*scan)\b")
    head_pat = re.compile(r"(?i)^(scann?\s*ed\s*with|escanead[oa]\s+con)\s*$")
    while i < len(lines):
        curr = lines[i].strip()
        if head_pat.match(curr) and i + 1 < len(lines):
            nxt = lines[i+1].strip()
            if join_pat.search(nxt):
                out.append(f"{curr} {nxt}")
                merged += 1
                i += 2
                continue
        out.append(lines[i])
        i += 1
    return "\n".join(out), merged

# ------- Normalizadores base -------
def normalize_quotes_spaces_with_counts(text: str) -> Tuple[str, Dict[str,int]]:
    counts = {}
    text = fix_text(text)
    text = text.replace("\u00AD", "").replace("\t", " ")
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text, nA = _apply_and_count(text, r"(?m)^[\-\_•·═\=]{3,}\s*$", "", flags=re.I)
    counts["ascii_art_removed"] = nA
    text, n = _apply_and_count(text, r"[ ]{2,}", " ")
    counts["collapse_spaces"] = n
    text, n5 = _apply_and_count(text, r"(\w)-\n(\w)", r"\1\2", flags=re.UNICODE)
    counts["join_hyphen_linebreak"] = n5
    text, n6 = _apply_and_count(text, r"\n{3,}", "\n\n")
    counts["collapse_blank_lines"] = n6
    text, n7 = _apply_and_count(text, r"([A-ZÁÉÍÓÚÑ])['’]([A-ZÁÉÍÓÚÑ])", r"\1 \2")
    counts["float_apostrophe_inside_caps"] = n7
    text, n8 = _apply_and_count(text, r"(?<=\b)[\"'`]{1,2}(?=[A-ZÁÉÍÓÚÑ])", "")
    counts["float_quote_before_cap"] = n8
    text, n9 = _apply_and_count(text, r"(?<=[A-ZÁÉÍÓÚÑ])[\"'`]{1,2}(?=\b)", "")
    counts["float_quote_after_cap"] = n9
    text, n10 = _apply_and_count(text, r"(?<=\s)[\"'`]{1,2}(?=[A-ZÁÉÍÓÚÑ])", "")
    counts["float_quote_space_cap"] = n10
    return text.strip(), counts

def address_number_symbol_fix(text: str) -> Tuple[str, int]:
    pat = re.compile(
        r"(?P<via>\b(?:Av\.?|Avenida|Jr\.?|Jir(?:ón|on)|Calle|Psje\.?|Pasaje|Urb\.?|Urbanización)\b)\s+"
        r"(?P<nom>[A-Za-zÁÉÍÓÚáéíóúñÑ0-9 .'\-]{2,})\s+"
        r"(?:N\s*O|No|Nº|O)\s*(?P<num>\d{1,6})\b",
        flags=re.I
    )
    count = 0
    def _repl(m):
        nonlocal count
        count += 1
        return f"{m.group('via')} {m.group('nom')} N° {m.group('num')}"
    return pat.sub(_repl, text), count

# ------- N° seguro por línea (omite sensibles) -------
def normalize_n_symbol_safely(text: str) -> Tuple[str, Dict[str,int]]:
    counts = {"basic":0, "NO_to_symbol":0, "NP_to_symbol":0, "ctx_NO":0, "ctx_NP":0, "Nro_to_symbol":0, "No_to_symbol":0}
    out = []
    for line in text.splitlines():
        if is_sensitive_line(line):
            out.append(line); continue
        s = line
        s, n1 = _apply_and_count(s, r"(?<!\w)N\s*[º°]\b", "N°")
        counts["basic"] += n1
        s, n2 = _apply_and_count(s, r"(?i)\bN\s*O\b(?!\s*\d)", "N°")
        counts["NO_to_symbol"] += n2
        s, n3 = _apply_and_count(s, r"(?i)\bN\s*P\b(?!\s*\d)", "N°")
        counts["NP_to_symbol"] += n3
        s, n3b = _apply_and_count(s, r"(?i)\bNo\b(?!\s*\d)", "N°")
        counts["No_to_symbol"] += n3b
        s, n4a = _apply_and_count(s, r"(?i)\bNro\.?\s*(?=\d)", "N° ")
        counts["Nro_to_symbol"] += n4a
        s, n4b = _apply_and_count(s, r"(?i)\bNo\.?\s*(?=\d)", "N° ")
        counts["No_to_symbol"] += n4b
        s, n4 = _apply_and_count(s, r"(?i)\b(DNI|Expediente|Resoluci(?:ón|on)(?:\s+Final)?|Papeleta|Oficio)\s+N\s*O\b(?!\s*\d)", r"\1 N°")
        counts["ctx_NO"] += n4
        s, n5 = _apply_and_count(s, r"(?i)\b(DNI|Expediente|Resoluci(?:ón|on)(?:\s+Final)?|Papeleta|Oficio)\s+N\s*P\b(?!\s*\d)", r"\1 N°")
        counts["ctx_NP"] += n5
        out.append(s)
    return "\n".join(out), counts

# ------- N° en encabezados -------
def normalize_n_in_doc_headers(text: str) -> Tuple[str, int]:
    rx = re.compile(
        r"""(?ix)
        \b
        (OFICIO(?:\s+\w+){0,6}?|
         RESOLUCI[ÓO]N(?:\s+\w+){0,8}?|
         RESOLUCION(?:\s+\w+){0,8}?)
        \s+N
        (?:\s*(?:[\?\u00B0º°o0]|p|P)|\s*(?:No\.?|Nro\.?)|(?:\s*P)?)?
        \s*(?=\d)
        """)
    count = 0
    out = []
    for line in (text or "").splitlines():
        s, n = re.subn(rx, r"\1 N° ", line)
        count += n
        out.append(s)
    return "\n".join(out), count

def header_like_np_no_fix(text: str) -> Tuple[str,int]:
    count = 0
    out = []
    hdr_pat = re.compile(r"\b(OFICIO|RESOLUCI[ÓO]N)\b", re.I)
    def is_header_like(s: str) -> bool:
        s2 = s.strip()
        return (len(s2) <= 80 and s2 == s2.upper() and hdr_pat.search(s2))
    for ln in (text or "").splitlines():
        if is_header_like(ln):
            s, n = re.subn(r"(?i)\bN\s*(?:P|O|0|\?)\b", "N°", ln)
            s, n2 = re.subn(r"(?i)\bN(?:o|ro)\.?\b", "N°", s)
            s, n3 = re.subn(r"(?i)\bN°(?=\d)", "N° ", s)
            count += (n + n2 + n3)
            out.append(s)
        else:
            out.append(ln)
    return "\n".join(out), count

def final_symbol_sanitizer(text: str) -> Tuple[str, int]:
    legal_ctx = re.compile(r"\b(OFICIO|RESOLUCI[ÓO]N|EXPEDIENTE|ORDENANZA|LEY|PAPELETA|OFICIO)\b", re.I)
    changed = 0
    out = []
    for ln in (text or "").splitlines():
        if legal_ctx.search(ln):
            s, n1 = re.subn(r"(?i)\bN\s*P\b", "N°", ln)
            out.append(s); changed += n1
        else:
            out.append(ln)
    return "\n".join(out), changed

# --- OCR post-fix helpers (tablas) ---
def ocr_general_resolucion_fix(s: str) -> str:
    s = re.sub(r"(?i)res[o0]l[uo]c[i1í][o0]n", "Resolución", s)
    s = re.sub(r"(?i)res[o0]l[uo]c[i1í][o0]nes", "Resoluciones", s)
    return s

def ocr_general_levantamiento_fix(s: str) -> str:
    s = re.sub(r"(?i)lev[•\.]?antamiento|tevent\.?rvento|teventamiento|lev\•ntamiento|levartamiento", "levantamiento", s)
    return s

# [v1.29 ADD] Fuzzy OCR corrections en tablas (nombres/cargos)
FUZZY_OCR_TARGETS = [
    "JOSE LUIS BUENDIA MANTURANO",
    "EJECUTOR COACTIVO",
    "SERVICIO DE ADMINISTRACIÓN TRIBUTARIA DE HUANCAYO",
]
def ocr_fuzzy_cell_fix(s: str, threshold: int = 90) -> str:
    if not s: return s
    cand = s.upper()
    best = fuzzprocess.extractOne(cand, FUZZY_OCR_TARGETS, scorer=fuzz.ratio)
    if best and best[1] >= threshold:
        return best[0]
    return s

def ocr_fix_table_cell(s: str) -> str:
    if not s: return s
    repl = {
        "ResolLEi6n": "Resolución",
        "Resohxi6n": "Resolución",
        "ResohRi6n": "Resolución",
        "Resoluci6n": "Resolución",
        "Resoluclón": "Resolución",
        "Resolucién": "Resolución",
        "ResoIución": "Resolución",
        "RES0LUCION": "RESOLUCIÓN",
        "tev•ntamiento": "levantamiento",
        "lev•ntamiento": "levantamiento",
        "tevent.rvento": "levantamiento",     # [v1.29 ADD] (según parche)
        "teventamiento": "levantamiento",
        "levartamiento": "levantamiento",
        "N•": "N°",
        "Nº": "N°",
        "8UENDtA": "BUENDIA",                 # [v1.29 ADD]
        "cmodmiento": "conocimiento",         # [v1.29 ADD]
        "NOIA": "NORIA",                      # [v1.29 ADD] (mejorable según contexto)
    }
    for a,b in repl.items():
        s = s.replace(a, b)
    s = ocr_general_resolucion_fix(s)
    s = ocr_general_levantamiento_fix(s)
    s = s.replace("•", "e")
    s = re.sub(r"(?<=\d)[Ss](?=\d)", "5", s)
    s = re.sub(r"(?<=\d)[Oo](?=\d)", "0", s)
    s = re.sub(r"(?<=\d)[Il](?=\d)", "1", s)
    # [v1.29 ADD] Fuzzy post-pass en celdas (sólo si se parece muuucho)
    s = ocr_fuzzy_cell_fix(s, threshold=92)
    return s

def detect_header_columns(header_cells: List[str]) -> Dict[str, List[int]]:
    cols = {"dni": [], "ruc": [], "resol": [], "monto": [], "cheque": [], "exped": []}
    for idx, h in enumerate(header_cells or []):
        hh = (h or "").strip().upper()
        if re.search(r"\bDNI\b|\bDNI\s*[/\-]\s*RUC\b", hh): cols["dni"].append(idx)
        if re.search(r"\bRUC\b", hh): cols["ruc"].append(idx)
        if re.search(r"RESOLU", hh): cols["resol"].append(idx)
        if re.search(r"\bMONTO\b|\bIMPORTE\b|\bTOTAL\b|\bS/?\b", hh): cols["monto"].append(idx)
        if re.search(r"\bCHEQUE\b|\bN[°º]?\s*DE\s*CHEQUE\b", hh): cols["cheque"].append(idx)
        if re.search(r"\bEXPEDIENTE\b", hh): cols["exped"].append(idx)
    return cols

def _compact_digits_inside(s: str) -> str:
    return re.sub(r"(?:(?<=\D)|^)(\d(?:[\s\.]?\d){5,})(?=(?:\D|$))",
                  lambda m: re.sub(r"[\s\.]+", "", m.group(1)), s)

# PII sólo en tablas etiquetadas
def mask_in_rows_by_cols(rows: List[List[str]], cols_map: Dict[str, List[int]], table_pii_masking: bool=False) -> Tuple[List[List[str]], Dict[str,int]]:
    counts = {"dni":0, "ruc":0}
    out = []

    def _group_digits_for_readability(s: str) -> str:
        def repl(m):
            digits = m.group(0)
            if 8 <= len(digits) <= 20:
                return " ".join(digits[i:i+4] for i in range(0, len(digits), 4))
            return digits
        return re.sub(r"\b\d{8,20}\b", repl, s)

    for row in rows:
        new_row = []
        for ci, cell in enumerate(row):
            val = ocr_fix_table_cell(cell or "")

            # PII opcional SOLO si está habilitado globalmente o si activaste TABLE_PII_ONLY
            do_pii = table_pii_masking
            if (do_pii or ENABLE_PII_MASKING) and ci in cols_map.get("dni", []):
                def _dni_mask(m):
                    counts["dni"] += 1
                    d = m.group(0)
                    return d[:4] + "****"
                val = re.sub(r"\b(\d{8})\b", _dni_mask, val)

            if (do_pii or ENABLE_PII_MASKING) and MASK_RUC and (ci in cols_map.get("ruc", [])):
                def _ruc_mask(m):
                    counts["ruc"] += 1
                    d = m.group(0)
                    return d[:4] + "*******"
                val = re.sub(r"\b(\d{11})\b", _ruc_mask, val)

            # Post-fix SOLO en columnas monetarias/cheque (nunca Expediente)
            if ci in (cols_map.get("monto", []) + cols_map.get("cheque", [])):
                val = _compact_digits_inside(val)
                val = re.sub(r"(?<=\d)Z(?=\d)", "2", val, flags=re.I)  # [v1.29 ADD]
                val = re.sub(r"(?<=\d)O(?=\d)", "0", val)
                val = re.sub(r"(?<=\d)I(?=\d)", "1", val)

            # Moneda en columnas MONTO (prefijo + formato)
            if ci in cols_map.get("monto", []):
                _t, _ = normalize_pen_currency_prefix(val)
                val2, _ = normalize_pen_currency_amounts(_t)
                val = val2

            # Cheques con sufijo numérico separado (p.ej. "... 60")
            if ci in cols_map.get("cheque", []):
                val = re.sub(r"\b(\d{6,})\s+(\d{2,3})\b", r"\1\2", val)
                val = _group_digits_for_readability(val)

            # Limpieza tipográfica en columnas de resolución
            if ci in cols_map.get("resol", []):
                val = re.sub(r"(?<=\d)[Ss](?=\d)", "5", val)
                val = re.sub(r"(?<=\d)[Oo](?=\d)", "0", val)
                val = re.sub(r"(?<=\d)[Il](?=\d)", "1", val)

            new_row.append(val)
        out.append(new_row)
    return out, counts

def _table_to_markdown(t: Table, t_index: int) -> Tuple[str, Dict]:
    # Primera pasada
    rows = []
    for r in t.rows:
        row = []
        for c in r.cells:
            row.append(" ".join(p.text.strip() for p in c.paragraphs if p.text).strip())
        rows.append(row)

    def row_is_empty_or_watermark(rr: List[str]) -> bool:
        joined = " ".join((rr or [])).strip()
        if not joined:
            return True
        return is_watermark_line(joined) or bool(WM_INLINE.search(joined))

    rows_filtered = [rr for rr in rows if not row_is_empty_or_watermark(rr)]
    use_rows = rows_filtered if any(any(c.strip() for c in r) for r in rows_filtered) else rows

    n_rows = len(use_rows)
    n_cols = max((len(r) for r in use_rows), default=0)
    if n_rows == 0 or n_cols == 0:
        return "", {"n_rows": n_rows, "n_cols": n_cols, "rows": use_rows, "markdown": "", "has_header": False, "pii_masked": {"dni":0,"ruc":0}, "source_index": t_index}

    header = use_rows[0]

    def looks_like_narrative_header(hcells: List[str]) -> bool:
        joined = " ".join((hcells or [])).strip()
        U = joined.upper()
        tab_kw = ("EXPEDIENTE","CHEQUE","MONTO","DNI","RUC","ORDENANTE","EJECUTOR","RESOLUCIÓN","IMPORTE","TOTAL")
        if any(k in U for k in tab_kw):
            return False
        if re.search(r"[.!?;:]\s*$", joined):
            return True
        if re.match(r"(?i)^\s*(SIENDO|POR CUANTO|VISTO|CONSIDERANDO)\b", joined):
            return True
        return len(joined) >= 200

    alpha_cells = sum(bool(re.search(r"[A-Za-zÁÉÍÓÚáéíóúñÑ]", c or "")) for c in header)
    has_header = (alpha_cells / max(1, len(header))) >= 0.4 and not looks_like_narrative_header(header)

    tab_kw = ("EXPEDIENTE","CHEQUE","MONTO","DNI","RUC","ORDENANTE","EJECUTOR","RESOLUCIÓN","IMPORTE","TOTAL")
    if not has_header:
        if sum(k in " ".join(header).upper() for k in tab_kw) >= 2:
            has_header = True

    # Filtro 2×2 ruido — mantener si parece ACTA/Notificación (v1.29)
    header_join = " ".join(header).upper()
    has_kw = any(k in header_join for k in tab_kw)
    small_noise = (n_rows <= 2 and n_cols <= 2 and not has_header and not has_kw)
    # [v1.29 ADD] whitelista claves de acta
    acta_keys = ("ACTA","D.N.I","DNI","NOTIFICADOR","FIRMA","NEGATIVA","RECEPCIÓN","RECEPCION")
    keep_small_acta = small_noise and any(any(k in (cell or "").upper() for k in acta_keys) for cell in sum(use_rows, []))
    if small_noise and not keep_small_acta:
        meta = {"n_rows": n_rows, "n_cols": n_cols, "rows": use_rows, "markdown": "", "has_header": False, "pii_masked": {"dni":0,"ruc":0}, "source_index": t_index, "discarded_small_noise": True, "discard_reason": "table_≤2x2_without_keywords"}
        return "", meta

    table_pii_counts = {"dni":0, "ruc":0}

    if has_header:
        cols_map = detect_header_columns(header)
        rows_fixed, counts = mask_in_rows_by_cols(use_rows, cols_map, table_pii_masking=(ENABLE_PII_MASKING or TABLE_PII_ONLY))
        use_rows = rows_fixed
        table_pii_counts["dni"] += counts["dni"]
        table_pii_counts["ruc"] += counts["ruc"]
        # dedup header repetido
        header_norm = tuple((c or "").strip().upper() for c in use_rows[0])
        dedup_rows = [use_rows[0]]
        for r in use_rows[1:]:
            if tuple((c or "").strip().upper() for c in r) == header_norm:
                continue
            dedup_rows.append(r)
        use_rows = dedup_rows
    else:
        use_rows = [[ocr_fix_table_cell(c) for c in r] for r in use_rows]

    md = []
    if has_header:
        md.append("| " + " | ".join(c or "" for c in use_rows[0]) + " |")
        md.append("| " + " | ".join("---" for _ in range(n_cols)) + " |")
        data_rows = use_rows[1:]
    else:
        md.append("| " + " | ".join(f"Col {i+1}" for i in range(n_cols)) + " |")
        md.append("| " + " | ".join("---" for _ in range(n_cols)) + " |")
        data_rows = use_rows

    for r in data_rows:
        cells = (r + [""] * n_cols)[:n_cols]
        md.append("| " + " | ".join(c or "" for c in cells) + " |")

    md_str = "\n".join(md)
    meta = {
        "n_rows": n_rows, "n_cols": n_cols, "rows": use_rows,
        "markdown": md_str, "has_header": has_header,
        "pii_masked": table_pii_counts, "source_index": t_index
    }
    if keep_small_acta:
        meta["kept_small_acta"] = True  # [v1.29 ADD]
    return md_str, meta

def extract_text_docx_with_tables(fbytes: bytes) -> Tuple[str, List[Dict]]:
    doc = Document(io.BytesIO(fbytes))
    md_parts = []
    tables_meta: List[Dict] = []
    t_index = 0
    for block in iter_block_items(doc):
        if isinstance(block, Paragraph):
            txt = block.text or ""
            md_parts.append(txt)
        else:
            md_tbl, meta_tbl = _table_to_markdown(block, t_index)
            t_index += 1
            if md_tbl.strip():
                md_parts.append(md_tbl)
            tables_meta.append(meta_tbl)
    full = "\n\n".join(p for p in md_parts if p is not None)
    return full, tables_meta

def iter_block_items(parent):
    body = parent.element.body
    for child in body.iterchildren():
        if isinstance(child, CT_P):
            yield Paragraph(child, parent)
        elif isinstance(child, CT_Tbl):
            yield Table(child, parent)

# ---- Limpieza de numeración de línea OCR ----
def remove_left_line_numbers(text: str) -> Tuple[str, int]:
    if not REMOVE_LINE_NUMBERS:
        return text, 0
    count = 0
    out = []
    for ln in text.splitlines():
        s, n = re.subn(r"^\s*(?:\d{1,3}[\)\.]|\d{1,3}\s+\|)\s+", "", ln)
        count += n
        out.append(s)
    return "\n".join(out), count

def unify_linebreaks(text: str) -> str:
    text = promote_inline_headers(text)
    lines = text.splitlines()
    out, buf = [], ""
    def is_header_or_list(s: str) -> bool:
        s2 = s.strip()
        if not s2: return False
        if re.match(r"^#{1,6}\s", s2): return True
        if s2.upper() in SECTION_HEADERS_UP: return True
        if re.match(r"^[\*\-•]\s+", s2): return True
        if len(s2) <= 80 and s2 == s2.upper() and any(c.isalpha() for c in s2):
            return True
        if s2.startswith("| ") and s2.endswith(" |"): return True
        return False
    for ln in lines:
        s = ln.strip()
        if not buf:
            buf = s; continue
        prev = buf
        if (not prev.endswith(('.', ':', ';', '¿', '?', '!', '…'))) and (not is_header_or_list(s)):
            buf = prev + " " + s
        else:
            out.append(prev); buf = s
    if buf: out.append(buf)
    return "".join(l+"\n" for l in out).rstrip("\n")

def promote_inline_headers(text: str) -> str:
    text = re.sub(r"(?i)\bDADO\s+CUENTA\s+Y\s+CONSIDERANDO\s*[:;]?", "\nCONSIDERANDO\n", text)
    text = re.sub(r"(?i)\bCONSIDERANDO\s*[:;]?", "\nCONSIDERANDO\n", text)
    text = re.sub(r"(?i)\bVISTO\s*[:;]?", "\nVISTO\n", text)
    text = re.sub(r"(?i)\bBASE\s+LEGAL\s*[:;]?", "\nBASE LEGAL\n", text)
    text = re.sub(r"(?i)\bSE\s+RESUELVE\s*[:;]?", "\nSE RESUELVE\n", text)
    text = re.sub(r"(?i)(?<!SE\s)\bRESUELVE\s*[:;]?", "\nRESUELVE\n", text)
    text = re.sub(r"(?i)\bANTECEDENTES\s*[:;]?", "\nANTECEDENTES\n", text)
    text = re.sub(r"(?i)\bFUNDAMENTOS?\s*[:;]?", "\nFUNDAMENTOS\n", text)
    text = re.sub(r"(?m)^\s*[•\*]\s+", "- ", text)
    return text

def fix_ocr_spacing_in_norms(text: str) -> str:
    text = re.sub(r"\s*-\s*", "-", text)
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\bM\s*T\s*C\b", "MTC", text)
    text = re.sub(r"\bE\s*S\s*T\s*A\s*D\s*O\b", "ESTADO", text)
    text = re.sub(r"(?<=\b)([A-Z])(?:\s)(?=[A-Z]\b)(?=[A-Z](?:\s[A-Z])*\s*[-\d])", "", text)
    text = re.sub(r"(?<=[A-Za-z0-9])[•·](?=[A-Za-z0-9])", "-", text)
    return text

def apply_common_fixes_with_counts(text: str) -> Tuple[str, int]:
    total = 0
    for pat, rep in COMMON_FIXES.items():
        text, n = re.subn(pat, rep, text, flags=re.I)
        total += n
    return text, total

def restore_accents_safe(text: str) -> str:
    def repl(m):
        w = m.group(0)
        if w.isupper():
            return w
        base = unidecode(w.lower())
        if base in ACCENT_FIXES:
            fixed = ACCENT_FIXES[base]
            return fixed.capitalize() if w[0].isupper() else fixed
        return w
    return re.sub(r"\b[A-Za-zÁÉÍÓÚáéíóúñÑ]{3,}\b", repl, text)

def dedup_trailing_lines(text: str) -> str:
    lines = [l.rstrip() for l in text.splitlines()]
    seen = set(); out = []
    for l in reversed(lines):
        up = l.strip().upper()
        if up in {"REGISTRESE", "REGÍSTRESE"}:
            if up in seen:
                continue
            seen.add(up)
        out.append(l)
    return "\n".join(reversed(out))

def strip_common_headers_footers(lines: List[str]) -> Tuple[List[str], Dict[str,int]]:
    cleaned, removed = [], 0
    inline_removed = 0
    for ln in lines:
        raw_ln = ln
        l = ln.strip()
        if not l:
            cleaned.append(ln); continue
        if re.search(r"p[aá]gina\s+\d+(\s+de\s+\d+)?", l, flags=re.I):
            removed += 1; continue
        if re.search(r"^https?://", l, flags=re.I):
            removed += 1; continue
        if is_watermark_line(l):
            removed += 1; continue
        l2, cnt = remove_watermark_substrings(l)
        inline_removed += cnt
        if not l2:
            removed += 1; continue
        if len(l2) <= 3 and re.match(r"^[\-\_•·]+$", l2):
            removed += 1; continue
        cleaned.append(l2 if l2 != l else raw_ln)
    return cleaned, {"line_removed": removed, "inline_removed": inline_removed}

def remove_scanner_watermarks(text: str) -> Tuple[str, Dict[str,int]]:
    if MERGE_SPLIT_WATERMARKS:
        text, merged = merge_split_watermark_lines(text)
    else:
        merged = 0
    lines = text.splitlines()
    new_lines, rm_counts = strip_common_headers_footers(lines)
    rm_counts["split_watermarks_merged"] = merged
    return "\n".join(new_lines), rm_counts

def dedup_institutional_headers(text: str) -> Tuple[str, int]:
    if not DEDUP_INSTITUTIONAL_HEADERS:
        return text, 0
    lines = text.splitlines()
    out = []
    seen = set()
    bic_pat = re.compile(r"(?i)^\s*!?\s*AÑO\s+DEL\s+BICENTENARIO.*PER[ÚU]\s*!?\s*$")
    for ln in lines:
        if bic_pat.match(ln.strip()):
            key = "BICENTENARIO_HDR"
            if key in seen:
                continue
            seen.add(key)
        out.append(ln)
    return "\n".join(out), len(lines) - len(out)

def dedup_uppercase_global(text: str) -> Tuple[str,int]:
    lines = text.splitlines()
    out, seen, removed = [], set(), 0
    for ln in lines:
        s = ln.strip()
        up = s.upper()
        if len(s) > 0 and s == up and len(s) <= 120 and any(c.isalpha() for c in s):
            if up in seen:
                removed += 1
                continue
            seen.add(up)
        out.append(ln)
    return "\n".join(out), removed

def dedup_consecutive_equal_uppercase(text: str) -> Tuple[str, int]:
    lines = text.splitlines()
    out = []
    dedup = 0
    prev_up = None
    for ln in lines:
        up = ln.strip().upper()
        if prev_up is not None and up == prev_up and len(up) <= 120 and up and up == up.upper():
            dedup += 1
        else:
            out.append(ln)
        prev_up = up
    return "\n".join(out), dedup

# ----- Compactación de dígitos SAFE -----
def compact_long_digit_blocks_safe(text: str) -> Tuple[str, int]:
    count_total = 0
    out = []
    rx = re.compile(r"(?:(?<=\D)|^)(\d(?:\s?\d){7,})(?=(?:\D|$))")
    for line in text.splitlines():
        if is_sensitive_line(line):
            out.append(line); continue
        def join_digits(m):
            nonlocal count_total
            raw = m.group(1)
            joined = re.sub(r"\s+", "", raw)
            if joined.isdigit() and len(joined) >= 8:
                count_total += 1
                return joined
            return raw
        out.append(rx.sub(join_digits, line))
    return "\n".join(out), count_total

# [v1.29 ADD] Compactación “tipo cheque” en texto, acotada por contexto
CHEQUE_LINE_HINT = re.compile(r"(?i)\bCHEQUE\b|\bN[º°]?\s*DE\s*CHEQUE\b")
def compact_cheque_like_sequences_text(text: str) -> Tuple[str,int]:
    cnt = 0
    out = []
    for ln in text.splitlines():
        if is_sensitive_line(ln) or not CHEQUE_LINE_HINT.search(ln or ""):
            out.append(ln); continue
        def _join(m):
            nonlocal cnt
            raw = m.group(0)
            j = re.sub(r"[\s\.]+", "", raw)
            if len(j) >= 12:
                cnt += 1
                return j
            return raw
        s = re.sub(r"\b(?:\d[\s\.]?){8,}\d\b", _join, ln)
        out.append(s)
    return "\n".join(out), cnt

# ----- Moneda -----
def normalize_pen_currency_prefix(text: str) -> Tuple[str, int]:
    count = 0
    patterns = [
        r"(?i)\bS\s*[\/.]?\s*(?=[0-9])",
        r"(?i)\bS\s*[ilI]\.?(?=\d)",
        r"(?i)\bSI\.\s*(?=\d)",
        r"(?i)\bS\.\s*(?=\d)",
        r"(?i)\bS\s*\\\s*(?=\d)",
    ]
    def repl(m):
        nonlocal count
        count += 1
        return "S/"
    txt = text
    for p in patterns:
        txt = re.sub(p, repl, txt)
    return txt, count

def normalize_pen_currency_amounts(text: str) -> Tuple[str, int]:
    if not FORMAT_CURRENCY_AMOUNTS:
        return text, 0
    count = 0
    def _fmt(m):
        nonlocal count
        prefix = m.group(1)
        raw = m.group(2)
        # [v1.29 ADD] Z/I/O entre dígitos en contextos monetarios
        raw = re.sub(r"(?<=\d)Z(?=\d)", "2", raw, flags=re.I)
        raw = re.sub(r"(?<=\d)O(?=[\.,]?\d{0,2}\b)", "0", raw)
        raw = re.sub(r"(?<=\d)I(?=[\.,]\d{2}\b)", "1", raw)
        raw = raw.strip()

        digits_only = re.sub(r"[^\d,\.]", "", raw)
        if re.search(r"\d{1,3}(?:\.\d{3})+,\d{2}$", digits_only):
            entero = re.sub(r"[^\d]", "", digits_only.rsplit(",",1)[0])
            dec = digits_only.rsplit(",",1)[1]
        else:
            only_digits = re.sub(r"[^\d]", "", raw)
            if len(only_digits) >= 3:
                entero, dec = (only_digits[:-2] or "0"), only_digits[-2:]
            elif len(only_digits) == 2:
                entero, dec = "0", only_digits
            else:
                return prefix + " " + raw

        try:
            entero_fmt = "{:,}".format(int(entero))
        except Exception:
            return prefix + " " + raw

        count += 1
        return f"{prefix} {entero_fmt}.{dec}"
    pat = re.compile(r"(S/)\s*([0-9\.,\sZI]+)")
    res = pat.sub(_fmt, text)

    def _fallback_plain_digits(m):
        nonlocal count
        digits = m.group(1)
        if len(digits) >= 4:
            count += 1
            entero_fmt = "{:,}".format(int(digits))
            return f"S/ {entero_fmt}.00"
        return "S/ " + digits
    res = re.sub(r"S/\s*(\d{4,})\b", _fallback_plain_digits, res)
    return res, count

# [v1.29 ADD] extracción sets de montos antes/después (para transform_log)
def _extract_money_set(text: str) -> List[str]:
    if not text: return []
    # capturar S/ 1.234,56 | S/. 1234.56 | S/ 1234 | SI. 1.234,56 (normalizar prefijo)
    t, _ = normalize_pen_currency_prefix(text)
    found = re.findall(r"S/\s*[0-9\.\,]{1,15}", t)
    # canon a #.##
    out = []
    for f in found:
        num = f.split("S/")[1].strip()
        # si trae . como miles y , como decimales → convertir
        if re.search(r"\d{1,3}(?:\.\d{3})+,\d{2}$", num):
            entero = re.sub(r"[^\d]", "", num.rsplit(",",1)[0])
            dec = num.rsplit(",",1)[1]
            out.append(f"{entero}.{dec}")
        else:
            only = re.sub(r"[^\d]", "", num)
            if only:
                if len(only) >= 3:
                    out.append(only[:-2] + "." + only[-2:])
                else:
                    out.append(only)
    # dedup
    dd = []
    for v in out:
        if v not in dd: dd.append(v)
    return dd

# ----- Guardia de integridad (números largos en líneas sensibles) -----
LONG_NUM_RX = re.compile(r"\b\d{8,}\b")
def restore_long_digit_tokens_from_original(orig_line: str, clean_line: str) -> Tuple[str, bool]:
    def non_monetary_long_tokens(line: str) -> List[Tuple[int,int,str]]:
        toks = []
        for m in LONG_NUM_RX.finditer(line):
            start, end = m.start(), m.end()
            prefix = line[max(0, start-3):start]
            if "S/" in prefix:
                continue
            toks.append((start, end, m.group(0)))
        return toks
    orig_tokens = non_monetary_long_tokens(orig_line)
    clean_tokens = list(LONG_NUM_RX.finditer(clean_line))
    clean_tokens = [(m.start(), m.end(), m.group(0)) for m in clean_tokens
                    if "S/" not in clean_line[max(0, m.start()-3):m.start()]]
    if not orig_tokens and not clean_tokens:
        return clean_line, False
    if len(orig_tokens) != len(clean_tokens):
        return orig_line, True
    new_line_parts = []
    last = 0
    changed = False
    for (c_start, c_end, c_tok), (_, _, o_tok) in zip(clean_tokens, orig_tokens):
        new_line_parts.append(clean_line[last:c_start])
        new_line_parts.append(o_tok)
        if o_tok != c_tok:
            changed = True
        last = c_end
    new_line_parts.append(clean_line[last:])
    return "".join(new_line_parts), changed

def integrity_guard_sensitive_lines(orig_text: str, cleaned_text: str) -> Tuple[str, int]:
    orig_lines = orig_text.splitlines()
    clean_lines = cleaned_text.splitlines()
    out, fixes = [], 0
    for i in range(max(len(orig_lines), len(clean_lines))):
        o = orig_lines[i] if i < len(orig_lines) else ""
        c = clean_lines[i] if i < len(clean_lines) else ""
        if is_sensitive_line(o) or is_sensitive_line(c):
            c2, changed = restore_long_digit_tokens_from_original(o, c)
            if changed: fixes += 1
            out.append(c2)
        else:
            out.append(c)
    return "\n".join(out), fixes

# ---------- PII (texto corrido) ----------
def mask_pii(text: str, mask_ruc: bool = False) -> Tuple[str, Dict[str, List[str]]]:
    found: Dict[str, List[str]] = {"dni": [], "placa": [], "telefono": [], "email": [], "direccion": [], "ruc": [], "ce": [], "pasaporte": []}
    if not ENABLE_PII_MASKING or TABLE_PII_ONLY:
        return text, found
    def _dni_ctx(m):
        dni = m.group(1)
        found["dni"].append(dni)
        return f"DNI N° {dni[:4]}****"
    text = re.sub(r"(?i)\bDNI\s*(?:N(?:[°º]|\s*O)?\s*)?(\d{8})\b", _dni_ctx, text)
    def _ce_mask(m):
        num = m.group(1); found["ce"].append(num); return f"CE N° {num[:3]}****"
    text = re.sub(r"(?i)\bCE\s*(?:N(?:[°º]|\s*O)?\s*)?(\d{6,9})\b", _ce_mask, text)
    def _pass_mask(m):
        num = m.group(1); found["pasaporte"].append(num); return f"PASAPORTE {num[:2]}***"
    text = re.sub(r"(?i)\bPASAPORTE\s*([A-Z0-9]{6,9})\b", _pass_mask, text)
    if mask_ruc:
        def _ruc_mask(m):
            ruc = m.group(1); found["ruc"].append(ruc); return ruc[:4] + "*******"
        text = re.sub(r"\b(\d{11})\b", _ruc_mask, text)
    def _placa_mask(m):
        placa = m.group(1); found["placa"].append(placa); return placa[:3] + "-***"
    text = re.sub(r"\b([A-Z]{3}-[0-9A-Z]{3})\b", _placa_mask, text)
    def _tel_mask(m):
        tel = m.group(0); found["telefono"].append(tel); return tel[:3] + "******"
    text = re.sub(r"\b(?:\+?51[-\s]?)?(9\d{8})\b", _tel_mask, text)
    def _mail_mask(m):
        em = m.group(0); found["email"].append(em); user, dom = em.split("@", 1); return user[:2] + "***@" + dom
    text = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", _mail_mask, text)
    def _dir_mask(m):
        found["direccion"].append(m.group(0))
        via = m.group("via"); nom = m.group("nom"); tail = m.group("tail") or ""
        return f"{via} {nom} N° ****{tail}"
    text = re.sub(
        r"(?P<via>\b(?:Av\.?|Avenida|Jr\.?|Jir(?:ón|on)|Calle|Psje\.?|Pasaje|Mz\.?|Manzana|Urb\.?|Urbanización)\b)\s+"
        r"(?P<nom>[A-Za-zÁÉÍÓÚáéíóúñÑ0-9 .'\-]{2,})\s+"
        r"(?:N(?:[°ºoO]|ro\.?)?\s*)?(?P<num>\d{1,5})"
        r"(?P<tail>[^\n,;:—]*)",
        _dir_mask, text, flags=re.I
    )
    return text, found

# --------- Metadatos / utilidades de extracción ---------
def find_first_date_iso(text: str) -> Optional[str]:
    candidates = re.findall(
        r"(\b\d{1,2}\s+de\s+[A-Za-záéíóúÁÉÍÓÚ]+\s+de(?:l)?\s+\d{4}\b|\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b\d{4}-\d{2}-\d{2}\b)",
        text, flags=re.I)
    for c in candidates:
        dt = dateparser.parse(c, languages=["es"])
        if dt: return dt.strftime("%Y-%m-%d")
    return None

def guess_entity(text: str) -> Optional[str]:
    T = text.upper()
    head = T[:2000]
    explicit = [
        (r"SERVICIO\s+DE\s+ADMINISTRACI[ÓO]N\s+TRIBUTARIA\s+DE\s+HUANCAYO", "SAT HUANCAYO"),
        (r"\bSAT\s*-?\s*HUANCAYO\b", "SAT HUANCAYO"),
        (r"\bSATH\b", "SATH"),
    ]
    for rx, label in explicit:
        if re.search(rx, head):
            return label
    best, best_score = None, 0
    for cand in DEFAULT_ENTITY_CANDIDATES:
        sc = fuzz.partial_ratio(cand.upper(), head)
        if sc > best_score:
            best, best_score = cand, sc
    return best if best_score >= 70 else None

def extract_norms(text: str) -> List[str]:
    t = text
    for pat, rep in NORM_ALIASES:
        t = re.sub(pat, rep, t, flags=re.I)
    norms = set()
    for g in re.finditer(r"\b(DS|DU|DL)\s+(?:N(?:[°º]|\s*O)?\s*)?0*(\d{1,4})\s*-\s*(\d{4})(?:-([A-Z]{2,6}))?\b", t, flags=re.I):
        kind = g.group(1).upper()
        num_raw = g.group(2)
        year = g.group(3)
        sigla = f"-{g.group(4)}" if g.group(4) else ""
        num_fmt = num_raw if len(num_raw) >= 3 else num_raw.zfill(3)
        norms.add(f"{kind} N° {num_fmt}-{year}{sigla}")
    for m in re.finditer(r"\bLEY\s+N(?:[°º]|\s*O)\s*([\d]{2,5}[A-Z\-]*)\b", t, flags=re.I):
        val = m.group(1).upper()
        val = re.sub(r"-LEY\b", "", val, flags=re.I)
        norms.add(f"LEY N° {val}")
    for m in re.finditer(r"\bORDENANZA\s+N(?:[°º]|\s*O)\s*([\d]{1,4}-\d{4}(?:-[A-Z]{2,6})?)\b", t, flags=re.I):
        norms.add(f"ORDENANZA N° {m.group(1).upper()}")
    return sorted(norms)

def _norm_expediente_token(e: str) -> str:
    e = re.sub(r"\b(\d{2})-(\d{3})(\d{6,})\b", r"\1-\2-\3", e)
    e = re.sub(r"\b(\d{2})(\d{3})-(\d{6,})\b", r"\1-\2-\3", e)
    return e

def extract_expediente(text: str) -> Optional[str]:
    t = text or ""
    m_ctx = re.search(r"(?i)\bEXPEDIENTE(?:\s+COACTIVO)?\s*N(?:[°º]|\s*O)?\s*([A-Z0-9\-\/\.]{8,})", t)
    if m_ctx:
        return _norm_expediente_token(m_ctx.group(1))
    for m in re.finditer(r"(?i)\bEXPEDIENTE(?:\s+COACTIVO)?\b", t):
        window = t[max(0, m.start()-60): m.end()+60]
        m_near = re.search(r"\b\d{2}-\d{3}-\d{6,}\b", window)
        if m_near:
            return _norm_expediente_token(m_near.group(0))
    m0 = re.search(r"\b\d{2}-\d{3}-\d{6,}\b", t)
    if m0:
        return _norm_expediente_token(m0.group(0))
    m2 = re.search(r"\b[A-Z]-\d{3,6}-\d{2}\b", t)
    return _norm_expediente_token(m2.group(0)) if m2 else None

def detect_tipo_doc(text: str) -> Optional[str]:
    head = "\n".join((text or "").splitlines()[:20])
    mof = re.search(r"(?i)\bOFICIO(?:\s+(?:MÚLTIPLE|CIRCULAR|ESPECIAL|M)|\s+\w+)*\b", head)
    if mof:
        return mof.group(0).upper()
    pats = [
        r"RESOLUCI[ÓO]N\s+DE\s+GERENCIA",
        r"RESOLUCI[ÓO]N\s+GERENCIAL",
        r"RESOLUCI[ÓO]N\s+JEFATURAL",
        r"RESOLUCI[ÓO]N\s+DIRECTORAL",
        r"RESOLUCI[ÓO]N\s+SUB\s*GERENCIAL",
    ]
    for p in pats:
        mm = re.search(p, text or "", flags=re.I)
        if mm:
            return mm.group(0).upper()
    if re.search(r"(?i)\bRESUELVE\b", text or ""):
        return "RESOLUCIÓN"
    return None

_SMALL_ES = {"de","del","la","las","el","los","y","e","o","u","en","para","por","con","a","al","un","una","unos","unas"}

def _smart_spanish_titlecase(s: str) -> str:
    if not s: return s
    m = re.search(r"\sN(?:[°º]|\s*O)\s*", s, flags=re.I)
    if m:
        head, tail = s[:m.start()], s[m.start():]
    else:
        head, tail = s, ""
    toks = re.findall(r"[A-Za-zÁÉÍÓÚáéíóúñÑ]+|[^A-Za-zÁÉÍÓÚáéíóúñÑ]+", head)
    out = []
    first_alpha_done = False
    for t in toks:
        if re.match(r"^[A-Za-zÁÉÍÓÚáéíóúñÑ]+$", t):
            tl = t.lower()
            if not first_alpha_done:
                out.append(tl.capitalize()); first_alpha_done = True
            else:
                out.append(tl if tl in _SMALL_ES else tl.capitalize())
        else:
            out.append(t)
    return "".join(out) + tail

def split_by_sections(text: str) -> List[Tuple[str, str]]:
    lines = text.splitlines()
    blocks, buf = [], []
    current_sec = "ENCABEZADO"
    header_regex = re.compile(r'^\s*(' + '|'.join(map(re.escape, SECTION_HEADERS_UP)) + r')\b', flags=re.I)
    def flush():
        nonlocal buf, current_sec
        if buf:
            blocks.append((current_sec, "\n".join(buf).strip()))
            buf = []
    for ln in lines:
        ln_stripped = ln.strip()
        m = header_regex.match(ln_stripped)
        if m:
            flush()
            current_sec = m.group(1).upper()
            rest = ln_stripped[m.end():].lstrip(" :;,—-")
            if rest:
                buf.append(rest)
            continue
        buf.append(ln)
    flush()
    return blocks

def quality_metrics(text: str, tables_meta: Optional[List[Dict]] = None) -> Dict[str, float]:
    text = text or ""
    total = max(1, len(text))
    symbol_ratio = sum(1 for c in text if not c.isalnum() and c not in " .,-;:()/[]\n|") / total
    tokens = text.split()
    def is_mostly_nonalpha(t: str) -> bool:
        if not t: return False
        alpha = sum(1 for c in t if t and c.isalpha())
        return alpha < max(1, int(0.5 * len(t)))
    nonalpha_ratio = (sum(1 for t in tokens if is_mostly_nonalpha(t)) / max(1, len(tokens)) if tokens else 0.0)
    table_count = len(tables_meta or [])
    line_count = text.count("\n") + 1 if text else 0
    return {
        "len_chars": len(text),
        "line_count": line_count,
        "table_count": table_count,
        "symbol_ratio": round(symbol_ratio, 3),
        "nonalpha_ratio": round(nonalpha_ratio, 3),
    }

# ---------- Diff de pérdidas ----------
def compute_loss_diffs(base_text: str, clean_text: str, slug: str) -> Dict[str, str]:
    diff = difflib.unified_diff(
        base_text.splitlines(),
        clean_text.splitlines(),
        fromfile=f"base_effective_{slug}",
        tofile=f"clean_{slug}",
        lineterm=""
    )
    diff_path = os.path.join(DIFF_DIR, f"{slug}.loss.diff.txt")
    diff_str = "\n".join(diff)
    with open(diff_path, "w", encoding="utf-8") as df:
        df.write(diff_str)
    return {"diff_path": diff_path, "has_diff": bool(diff_str.strip())}

# ---------- Núcleo ----------
def _doc_core_properties(doc: Document) -> Dict[str, Optional[str]]:
    cp = getattr(doc, "core_properties", None)
    if not cp: return {}
    meta = {}
    try:
        if cp.author:   meta["doc_author"] = cp.author
        if cp.title:    meta["doc_title_core"] = cp.title
        if cp.category: meta["doc_category"] = cp.category
        if cp.subject:  meta["doc_subject"] = cp.subject
        if cp.created:  meta["doc_created"] = cp.created.isoformat()
        if cp.modified: meta["doc_modified"] = cp.modified.isoformat()
    except Exception:
        pass
    return meta

# [v1.29 ADD] Golden tokens opcionales (no abortan, sólo registran)
GOLDEN_TOKENS_OPTIONAL = [
    "OFICIO MÚLTIPLE N° 08-013-000096330",
    "Huancayo, 19 de setiembre de 2024",
    "TORRES CAINICELA, ROLY LIDMAN",
    "S/ 1,382.96",
    "08-009-000436917"
]
def run_golden_tests(text: str) -> Dict[str, List[str]]:
    passed, failed = [], []
    U = text.upper()
    for tok in GOLDEN_TOKENS_OPTIONAL:
        if tok.upper() in U:
            passed.append(tok)
        else:
            failed.append(tok)
    return {"passed": passed, "failed": failed}

def process_file(fname: str, fbytes: bytes) -> Dict:
    ext = os.path.splitext(fname)[1].lower()
    if ext not in ACCEPTED_EXTS:
        raise ValueError(f"Archivo no permitido ({ext}). Solo se acepta .docx")

    log(f"Procesando: {fname}", EMOJI_INFO)
    file_hash = sha256_bytes(fbytes)

    # Extracción
    doc = Document(io.BytesIO(fbytes))
    raw_with_tables_md, tables_meta = extract_text_docx_with_tables(fbytes)
    orig_len = len(raw_with_tables_md or "")

    # Copia para guardia (antes de tocar números)
    guard_baseline = nfkc(raw_with_tables_md)
    if REMOVE_LINE_NUMBERS:
        guard_baseline, _rmn = remove_left_line_numbers(guard_baseline)
    guard_baseline = unify_linebreaks(guard_baseline)

    # Guardar baseline sidecar
    if INCLUDE_BASELINE_RAW:
        base_name = os.path.splitext(os.path.basename(fname))[0]
        slug_raw = slugify(base_name)
        raw_path = os.path.join(RAW_BASELINE_DIR, f"{slug_raw}.raw.txt")
        with open(raw_path, "w", encoding="utf-8") as rf:
            rf.write(guard_baseline)

    # Baseline efectivo para ratio
    base_for_loss, _wm0 = remove_scanner_watermarks(guard_baseline)
    base_for_loss, _dedup0 = dedup_institutional_headers(base_for_loss)
    base_for_loss, _dupG0 = dedup_uppercase_global(base_for_loss)
    base_for_loss, _dupUC0 = dedup_consecutive_equal_uppercase(base_for_loss)
    base_len_effective = len(base_for_loss or "")

    # Limpieza y normalización
    txt = nfkc(raw_with_tables_md)
    txt, norm_counts = normalize_quotes_spaces_with_counts(txt)
    txt, address_num_fixes = address_number_symbol_fix(txt)

    if REMOVE_LINE_NUMBERS:
        txt, removed_line_numbers = remove_left_line_numbers(txt)
    else:
        removed_line_numbers = 0

    txt = unify_linebreaks(txt)
    txt = fix_ocr_spacing_in_norms(txt)
    txt, common_fixes_count = apply_common_fixes_with_counts(txt)

    # N° seguro + encabezados
    txt, n_symbol_counts = normalize_n_symbol_safely(txt)
    txt, n_symbol_headers = normalize_n_in_doc_headers(txt)
    txt, header_like_fix_count = header_like_np_no_fix(txt)
    txt, final_np_fixes = final_symbol_sanitizer(txt)

    # Dígitos SAFE
    txt, digits_compacted = compact_long_digit_blocks_safe(txt)

    # [v1.29 ADD] Compactación robusta “tipo cheque” en texto (contextual)
    txt, cheque_compacted = compact_cheque_like_sequences_text(txt)

    # Guardia de integridad (restaura token exacto en líneas sensibles)
    txt, integrity_fixes = integrity_guard_sensitive_lines(guard_baseline, txt)

    # [v1.29 ADD] Snapshot de montos ANTES de normalizar moneda (para tracking)
    money_before_set = _extract_money_set(txt)

    # Moneda (en texto)
    txt, currency_prefix_count = normalize_pen_currency_prefix(txt)
    txt, currency_amounts_count = normalize_pen_currency_amounts(txt)

    # Watermarks + dedup institucional + dedups
    txt, wm_counts = remove_scanner_watermarks(txt)
    txt, inst_dedup = dedup_institutional_headers(txt)
    txt, uc_global_dedup = dedup_uppercase_global(txt)
    txt, uc_dedup = dedup_consecutive_equal_uppercase(txt)

    # Tildes + trailing
    txt = restore_accents_safe(txt)
    txt = dedup_trailing_lines(txt)

    # PII (texto corrido) — respeta TABLE_PII_ONLY
    txt, pii_found = mask_pii(txt, mask_ruc=MASK_RUC)

    # Metadatos
    fecha_iso = find_first_date_iso(txt)
    entidad = guess_entity(txt)
    tipo_doc = detect_tipo_doc(txt)
    normas = extract_norms(txt)
    expediente = extract_expediente(txt)

    # Título / número
    mres = re.search(r"(RESOLUCI[ÓO]N(?:\s+DE\s+[A-ZÁÉÍÓÚÑ ]{2,})?\s+N(?:[°º]|\s*O)\s*[\w\-\./]+)", txt, flags=re.I)
    numero_resolucion = None
    title = None
    if mres:
        title_raw = mres.group(1).strip()
        parts = re.split(r"\s+", title_raw)
        if parts: numero_resolucion = parts[-1]
        title = _smart_spanish_titlecase(title_raw)
    else:
        mof = re.search(r"(OFICIO(?:\s+\w+){0,6}?\s+N(?:[°º]|\s*O)\s*[\w\-\./]+)", txt, flags=re.I)
        if mof:
            title_raw = mof.group(1).strip()
            parts = re.split(r"\s+", title_raw)
            if parts: numero_resolucion = parts[-1]
            title = _smart_spanish_titlecase(title_raw)
        else:
            fl = [l.strip() for l in txt.splitlines()[:25] if l.strip()]
            caps = [l for l in fl if l == l.upper() and len(l) >= 10]
            base_name = os.path.basename(fname)
            title = _smart_spanish_titlecase(caps[0]) if caps else base_name

    # Secciones y offsets
    sections = split_by_sections(txt)
    section_spans = []
    cursor = 0
    for sec_name, sec_text in sections:
        idx = txt.find(sec_text, cursor)
        if idx == -1:
            idx = txt.find(sec_text)
        if idx == -1:
            section_spans.append({"name": sec_name, "start": None, "end": None})
        else:
            section_spans.append({"name": sec_name, "start": idx, "end": idx + len(sec_text)})
            cursor = idx + len(sec_text)

    sec_names = {s for s, _ in sections}
    has_visto = any(s.startswith("VISTO") for s in sec_names)
    has_considerando = any(s.startswith("CONSIDERANDO") for s in sec_names)
    has_resuelve = any(s.startswith("SE RESUELVE") or s.startswith("RESUELVE") for s in sec_names)

    doc_meta = _doc_core_properties(doc)

    # [v1.29 ADD] Snapshot de montos DESPUÉS de normalizar
    money_after_set = _extract_money_set(txt)

    # [v1.29 ADD] Golden tests opcionales
    golden = run_golden_tests(txt)

    base_meta = {
        "archivo": fname,
        "sha256": file_hash,
        "pipeline_version": PIPELINE_VERSION,
        **doc_meta,
        **({"entidad": entidad} if entidad else {}),
        **({"fecha_iso": fecha_iso} if fecha_iso else {}),
        **({"tipo_doc": tipo_doc} if tipo_doc else {}),
        **({"numero_resolucion": numero_resolucion} if numero_resolucion else {}),
        **({"expediente": expediente} if expediente else {}),
        **({"norma": normas} if normas else {}),
        "dni_masked": len(pii_found.get("dni", [])),
        "placas_masked": len(pii_found.get("placa", [])),
        "emails_masked": len(pii_found.get("email", [])),
        "telefonos_masked": len(pii_found.get("telefono", [])),
        "direcciones_masked": len(pii_found.get("direccion", [])),
        "ce_masked": len(pii_found.get("ce", [])),
        "pasaportes_masked": len(pii_found.get("pasaporte", [])),
        "has_visto": has_visto,
        "has_considerando": has_considerando,
        "has_resuelve": has_resuelve,
    }

    text_clean = txt
    clean_len = len(text_clean or "")
    len_ratio = (clean_len / orig_len) if orig_len else 1.0
    effective_len_ratio = (clean_len / base_len_effective) if base_len_effective else 1.0
    loss_alert = bool(base_len_effective and effective_len_ratio < STRICT_NO_LOSS_RATIO)

    loss_diff_info = {"diff_path": None, "has_diff": False}
    if loss_alert:
        log(f"Posible pérdida: ratio efectivo {effective_len_ratio:.3f} < {STRICT_NO_LOSS_RATIO}", EMOJI_WARN)
        base_name = os.path.splitext(os.path.basename(fname))[0]
        slug_tmp = slugify(base_name)
        loss_diff_info = compute_loss_diffs(base_for_loss, text_clean, slug_tmp)
        if ABORT_ON_LOSS:
            raise RuntimeError("Abortado por pérdida de longitud efectiva bajo el umbral.")

    # Armar JSON
    base = os.path.splitext(os.path.basename(fname))[0]
    slug = slugify(base)
    json_path = os.path.join(JSON_DIR, f"{slug}.clean.json")

    small_noise_discarded = sum(1 for tm in tables_meta if tm.get("discarded_small_noise"))

    out = {
        "schema_version": "clean-v1.29",   # (antes: clean-v1.28)
        "doc_id": slug,
        "filename": os.path.basename(fname),
        "sha256": file_hash,
        "pipeline_version": PIPELINE_VERSION,
        "source_ext": "docx",
        "title": title,
        "text_clean": text_clean,
        "sections": [{"name": s, "text": t} for s, t in sections],
        "section_spans": section_spans,
        "tables": tables_meta,   # {n_rows, n_cols, rows, markdown, has_header, pii_masked, source_index, discarded_small_noise?, kept_small_acta?}
        "metadata": base_meta,
        "pii_counts": {k: len(v) for k, v in pii_found.items()},
        "quality": quality_metrics(text_clean, tables_meta),
        "loss_check": {
            "orig_len": orig_len,
            "base_len_effective": base_len_effective,
            "clean_len": clean_len,
            "len_ratio_raw": round(len_ratio, 3),
            "len_ratio_effective": round(effective_len_ratio, 3),
            "alert_below_threshold": loss_alert,
            "threshold": STRICT_NO_LOSS_RATIO,
            **loss_diff_info
        },
        "transform_log": {
            "normalize_counts": norm_counts,
            "address_number_symbol_fixes": address_num_fixes,
            "removed_line_numbers": removed_line_numbers,
            "common_fixes_total": common_fixes_count,
            "n_symbol_safely_counts": n_symbol_counts,
            "n_symbol_headers_fixes": n_symbol_headers,
            "header_like_np_no_fixes": header_like_fix_count,
            "final_np_symbol_fixes": final_np_fixes,
            "digits_compacted_safe": digits_compacted,
            "cheque_compacted_text": cheque_compacted,     # [v1.29 ADD]
            "integrity_guard_fixes": integrity_fixes,
            "watermarks_removed_lines": wm_counts.get("line_removed", 0),
            "watermarks_removed_inline": wm_counts.get("inline_removed", 0),
            "watermarks_split_merged": wm_counts.get("split_watermarks_merged", 0),
            "institutional_headers_deduped": inst_dedup,
            "uppercase_global_deduped": uc_global_dedup,
            "uppercase_consecutive_deduped": uc_dedup,
            "currency_prefix_normalized": currency_prefix_count,
            "currency_amounts_formatted": currency_amounts_count,
            "enable_pii_masking": ENABLE_PII_MASKING,
            "mask_ruc": MASK_RUC,
            "table_pii_only": TABLE_PII_ONLY,
            "tables_pii_masked": [tm.get("pii_masked", {"dni":0,"ruc":0}) for tm in tables_meta],
            "small_tables_discarded": small_noise_discarded,
            # [v1.29 ADD] tracking montos antes/después
            "money_raw_set": money_before_set,
            "money_norm_set": money_after_set,
            # [v1.29 ADD] golden tests
            "golden_tests": golden
        }
    }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    log(f"Generado JSON: {json_path}", EMOJI_OK)
    return {"slug": slug, "json_path": json_path, "meta": base_meta}

def process_many(uploaded: Dict[str, bytes]) -> List[Dict]:
    results = []
    for fname, fbytes in uploaded.items():
        ext = os.path.splitext(fname)[1].lower()
        if ext not in ACCEPTED_EXTS:
            print(f"[SKIP] {fname}: Extensión '{ext}' no permitida. Solo .docx")
            continue
        try:
            r = process_file(fname, fbytes)
            results.append(r)
        except Exception as e:
            print(f"[ERROR] {fname}: {e}")
    return results

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    pass

if IN_COLAB:
    print(f"{EMOJI_INFO} Sube tus archivos Word (.docx). NO se permiten .zip, .pdf u otros. Puedes seleccionar varios a la vez.")
    up = files.upload()
    results = process_many(up)
    print("\nResumen:")
    for r in results:
        print(f"- {r['slug']}")
        print(f"  JSON: {r['json_path']}")
    print("\nDescarga manual:")
    print(" - Ve a la barra lateral de archivos (icono de carpeta) y navega a /content/output/clean_json")
    print(" - O ejecuta: from google.colab import files; files.download('<ruta_al_json>')")
else:
    print(f"{EMOJI_INFO} Entorno local detectado. OUT:", JSON_DIR)
    print("Para probar en código:\nwith open('/ruta/al/doc.docx','rb') as f: process_file('doc.docx', f.read())")


ℹ️ Sube tus archivos Word (.docx). NO se permiten .zip, .pdf u otros. Puedes seleccionar varios a la vez.


Saving CamScanner 9-26-25 17.11(1) (1).docx to CamScanner 9-26-25 17.11(1) (1) (1).docx
Saving gabi 1.docx to gabi 1 (1).docx
Saving gabi 3.docx to gabi 3 (1).docx
Saving gabi 4.docx to gabi 4 (1).docx
Saving gabi 5.docx to gabi 5 (1).docx
Saving gagi 2.docx to gagi 2 (1).docx
ℹ️ Procesando: CamScanner 9-26-25 17.11(1) (1) (1).docx
✅ Generado JSON: /content/output/clean_json/camscanner-9-26-25-17111-1-1.clean.json
ℹ️ Procesando: gabi 1 (1).docx
✅ Generado JSON: /content/output/clean_json/gabi-1-1.clean.json
ℹ️ Procesando: gabi 3 (1).docx
✅ Generado JSON: /content/output/clean_json/gabi-3-1.clean.json
ℹ️ Procesando: gabi 4 (1).docx
✅ Generado JSON: /content/output/clean_json/gabi-4-1.clean.json
ℹ️ Procesando: gabi 5 (1).docx
✅ Generado JSON: /content/output/clean_json/gabi-5-1.clean.json
ℹ️ Procesando: gagi 2 (1).docx
✅ Generado JSON: /content/output/clean_json/gagi-2-1.clean.json

Resumen:
- camscanner-9-26-25-17111-1-1
  JSON: /content/output/clean_json/camscanner-9-26-25-17111-1-1.cl